# 🚀 Enterprise Voice RAG ChatBot — Google Colab GPU Backend
Run this notebook on Google Colab with **T4 GPU** runtime for sub-3 second voice response latency.

**Instructions**:
1. Go to **Runtime ➔ Change runtime type ➔ T4 GPU**.
2. Run **Cell 1** to clone/update the repository.
3. Run **Cell 2** to install GPU PyTorch & backend dependencies.
4. Run **Cell 3** to set your API keys and launch the backend server with ngrok.

In [2]:
import os
# 1. Move to /content root
%cd /content

# Clean up nested clones if any exist
if os.path.exists('/content/Voice_ChatBot/Voice_ChatBot'):
    !rm -rf /content/Voice_ChatBot/Voice_ChatBot

# Clone repository if not present, else pull latest updates
if not os.path.exists('/content/Voice_ChatBot'):
    !git clone https://github.com/FENGFANCHEN-012/Voice_ChatBot.git
    %cd /content/Voice_ChatBot
else:
    %cd /content/Voice_ChatBot
    !git pull

print('✅ Repository is ready at /content/Voice_ChatBot')

/content
Cloning into 'Voice_ChatBot'...
remote: Enumerating objects: 421, done.
remote: Counting objects: 100% (421/421), done.
remote: Compressing objects: 100% (270/270), done.
remote: Total 421 (delta 180), reused 379 (delta 138), pack-reused 0 (from 0)
Receiving objects: 100% (421/421), 19.27 MiB | 20.50 MiB/s, done.
Resolving deltas: 100% (180/180), done.
/content/Voice_ChatBot
✅ Repository is ready at /content/Voice_ChatBot


In [3]:
# 2. Install PyTorch CUDA GPU & Backend Dependencies
%cd /content/Voice_ChatBot

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install -r backend/requirements.txt -q
!pip install pyngrok rank-bm25 llama-index-embeddings-huggingface -q

print('✅ All GPU dependencies installed successfully!')

/content/Voice_ChatBot
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━

In [ ]:
import os
from pyngrok import ngrok
from google.colab import userdata

# 1. Patch config.py in Colab
config_path = '/content/Voice_ChatBot/backend/app/config.py'
if os.path.exists(config_path):
    patched_config = 'from pydantic_settings import BaseSettings\n\n\nclass Settings(BaseSettings):\n    gemini_api_key: str = ""\n    \n    # LLM Provider: "gemini" (default), "deepseek", or "auto"\n    llm_provider: str = "deepseek"\n    deepseek_api_key: str = ""\n    deepseek_base_url: str = "https://api.deepseek.com"\n    deepseek_model_name: str = "deepseek-chat"\n    \n    Qdrant_api_key: str = ""\n    \n    host: str = "0.0.0.0"\n    port: int = 8000\n    cors_origins: str = "http://localhost:5173"\n    whisper_model_size: str = "base"\n    whisper_use_gpu: bool = True\n    tts_voice: str = "en-US-AndrewMultilingualNeural"\n    tts_rate: str = "+0%"\n    tts_pitch: str = "+0Hz"\n    \n    # ------------------------------------------\n    \n    # fast embedding model\n    embedding_model_name: str = "BAAI/bge-m3"\n    \n    \n    reranker_model_name: str = "BAAI/bge-reranker-v2-m3"\n\n    # Vector store — "chroma" (default) or "faiss"\n    vector_store_type: str = "chroma"\n    chroma_db_path: str = "chroma_db"\n    faiss_index_path: str = "faiss_index/index.faiss"\n    faiss_metadata_path: str = "faiss_index/metadata.pkl"\n\n    upload_dir: str = "uploads"\n    max_file_size_mb: int = 40\n    chunk_size: int = 512\n    chunk_overlap: int = 64\n    retrieval_top_k: int = 10\n    retrieval_fetch_k: int = 15\n    reranker_top_k: int = 5\n\n    model_config = {"env_file": ".env", "env_file_encoding": "utf-8", "extra": "ignore"}\n\n\nsettings = Settings()\n'
    with open(config_path, 'w', encoding='utf-8') as f:
        f.write(patched_config)
    print('✅ Verified and patched config.py in Colab')

# 2. Patch llm.py in Colab
llm_path = '/content/Voice_ChatBot/backend/app/pipeline/llm.py'
if os.path.exists(llm_path):
    patched_llm = 'import asyncio\nfrom loguru import logger\nimport google.generativeai as genai\nfrom app.pipeline.rate_limiter import gemini_rate_limiter\n\n\nSTATIC_PROMPT = """You are an enterprise policy assistant. Answer the user\'s question based ONLY on the provided context and conversation history.\n\nRules:\n1. Cite specific references from the context: chapter numbers, error codes (e.g. ERR-SSO-4039), form numbers (e.g. Form HR-PAY-102), directive names, and policy codes.\n2. When a cross-domain dependency exists (e.g. "See Chapter 3"), mention it explicitly.\n3. For step-by-step procedures, list them in order.\n4. If the question is a follow-up from conversation history (e.g. "explain more", "what do you mean"), answer from history context.\n5. If the question clearly requires document context that isn\'t available, state: "I cannot find this information in the uploaded documents."\n6. Do NOT hallucinate or make up information not present in the context.\n7. Keep answers concise but complete — include specific numbers, time limits, and thresholds when mentioned in context.\n\nBelow is the conversation history, document context, and the question."""\n\n\nimport json\nimport httpx\nfrom app.config import settings\n\n\nclass LLMClient:\n    def __init__(self, api_key: str):\n        if api_key:\n            genai.configure(api_key=api_key)\n        self.fallback_models = ["gemini-2.0-flash-lite", "gemini-1.5-flash", "gemini-2.0-flash"]\n        self.model_name = self.fallback_models[0]\n        self.last_rate_limit_wait: float = 0.0\n\n    def _build_dynamic_prompt(self, query: str, context: list[str], history: list[dict] | None = None) -> str:\n        docs = "\\n\\n".join(f"---\\n{c}" for c in context)\n\n        history_block = ""\n        if history:\n            lines = []\n            for msg in history[-6:]:\n                role = "User" if msg["role"] == "user" else "Assistant"\n                lines.append(f"{role}: {msg[\'content\']}")\n            history_block = "\\n".join(lines) + "\\n\\n"\n\n        return f"""Conversation History:\n{history_block}\nContext:\n{docs}\n\nQuestion: {query}"""\n\n    async def _generate_deepseek(self, dynamic: str) -> str:\n        api_key = settings.deepseek_api_key or settings.gemini_api_key\n        url = f"{settings.deepseek_base_url.rstrip(\'/\')}/chat/completions"\n        headers = {\n            "Authorization": f"Bearer {api_key}",\n            "Content-Type": "application/json"\n        }\n        payload = {\n            "model": settings.deepseek_model_name,\n            "messages": [\n                {"role": "system", "content": STATIC_PROMPT},\n                {"role": "user", "content": dynamic}\n            ],\n            "temperature": 0.3\n        }\n        async with httpx.AsyncClient(timeout=60.0) as client:\n            resp = await client.post(url, headers=headers, json=payload)\n            resp.raise_for_status()\n            data = resp.json()\n            return data["choices"][0]["message"]["content"]\n\n    async def _generate_deepseek_stream(self, dynamic: str):\n        api_key = settings.deepseek_api_key or settings.gemini_api_key\n        url = f"{settings.deepseek_base_url.rstrip(\'/\')}/chat/completions"\n        headers = {\n            "Authorization": f"Bearer {api_key}",\n            "Content-Type": "application/json"\n        }\n        payload = {\n            "model": settings.deepseek_model_name,\n            "messages": [\n                {"role": "system", "content": STATIC_PROMPT},\n                {"role": "user", "content": dynamic}\n            ],\n            "stream": True,\n            "temperature": 0.3\n        }\n        async with httpx.AsyncClient(timeout=60.0) as client:\n            async with client.stream("POST", url, headers=headers, json=payload) as resp:\n                resp.raise_for_status()\n                async for line in resp.aiter_lines():\n                    if line.startswith("data: "):\n                        data_str = line[6:].strip()\n                        if data_str == "[DONE]":\n                            break\n                        try:\n                            data = json.loads(data_str)\n                            delta = data["choices"][0]["delta"].get("content", "")\n                            if delta:\n                                yield delta\n                        except Exception:\n                            continue\n\n    async def generate(self, query: str, context: list[str], history: list[dict] | None = None) -> str:\n        dynamic = self._build_dynamic_prompt(query, context, history)\n        self.last_rate_limit_wait = 0.0\n\n        provider = settings.llm_provider.lower()\n        if provider == "deepseek" or (provider == "auto" and settings.deepseek_api_key):\n            try:\n                logger.info("[LLM] Generating answer using DeepSeek V3...")\n                return await self._generate_deepseek(dynamic)\n            except Exception as e:\n                logger.error(f"[LLM] DeepSeek failed ({e})")\n                if not settings.gemini_api_key:\n                    return f"DeepSeek API Error: {e}"\n                logger.warning("[LLM] Falling back to Gemini...")\n\n        if not settings.gemini_api_key:\n            return "Error: No valid LLM API key configured (neither DeepSeek nor Gemini)."\n\n        for model_name in self.fallback_models:\n            model = genai.GenerativeModel(model_name, system_instruction=STATIC_PROMPT)\n            for attempt in range(2):\n                try:\n                    waited = await gemini_rate_limiter.acquire()\n                    self.last_rate_limit_wait += waited\n                    response = await asyncio.to_thread(model.generate_content, dynamic)\n                    return response.text\n                except Exception as e:\n                    if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):\n                        logger.warning(f"[LLM] Quota hit on {model_name}, trying fallback model...")\n                        break\n                    else:\n                        raise\n        logger.error("[LLM] Quota exceeded on all fallback models")\n        return "I\'m experiencing high demand. Please try again in a minute."\n\n    async def generate_stream(self, query: str, context: list[str], history: list[dict] | None = None):\n        dynamic = self._build_dynamic_prompt(query, context, history)\n\n        provider = settings.llm_provider.lower()\n        if provider == "deepseek" or (provider == "auto" and settings.deepseek_api_key):\n            try:\n                logger.info("[LLM] Streaming answer using DeepSeek V3...")\n                async for chunk in self._generate_deepseek_stream(dynamic):\n                    yield chunk\n                return\n            except Exception as e:\n                logger.error(f"[LLM] DeepSeek streaming failed ({e})")\n                if not settings.gemini_api_key:\n                    yield f"DeepSeek API Error: {e}"\n                    return\n                logger.warning("[LLM] Falling back to Gemini...")\n\n        if not settings.gemini_api_key:\n            yield "Error: No valid LLM API key configured (neither DeepSeek nor Gemini)."\n            return\n\n        for model_name in self.fallback_models:\n            model = genai.GenerativeModel(model_name, system_instruction=STATIC_PROMPT)\n            for attempt in range(2):\n                try:\n                    await gemini_rate_limiter.acquire()\n                    response = await asyncio.to_thread(model.generate_content, dynamic, stream=True)\n                    for chunk in response:\n                        if chunk.text:\n                            yield chunk.text\n                    return\n                except Exception as e:\n                    if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):\n                        logger.warning(f"[LLM] Quota hit on {model_name}, trying fallback model...")\n                        break\n                    else:\n                        raise\n        logger.error("[LLM] Quota exceeded on all fallback models")\n        yield "I\'m experiencing high demand. Please try again in a minute."\n'
    with open(llm_path, 'w', encoding='utf-8') as f:
        f.write(patched_llm)
    print('✅ Verified and patched llm.py in Colab')

# 3. Patch audio_service.py in Colab
audio_path = '/content/Voice_ChatBot/backend/app/services/audio_service.py'
if os.path.exists(audio_path):
    patched_audio = 'import asyncio\nimport io\nimport os\nimport re\nimport hashlib\nimport tempfile\nfrom pathlib import Path\n\nimport edge_tts\nfrom faster_whisper import WhisperModel\nfrom app.config import settings\n\n_ffmpeg_dirs = [\n    str(Path(__file__).resolve().parent.parent.parent.parent / "venv" / "Lib" / "site-packages" / "imageio_ffmpeg" / "binaries"),\n    str(Path(__file__).resolve().parent.parent.parent.parent / ".venv" / "Lib" / "site-packages" / "imageio_ffmpeg" / "binaries"),\n]\nfor d in _ffmpeg_dirs:\n    if os.path.isdir(d):\n        os.environ["PATH"] = d + os.pathsep + os.environ.get("PATH", "")\n        break\n\n\ndef clean_text_for_tts(text: str) -> str:\n    # Replace slashes between word characters or standalone slashes with spaces so TTS doesn\'t say "slash"\n    text = re.sub(r\'(?<=\\w)/(?=\\w)\', \' \', text)\n    text = re.sub(r\'[/\\\\#*_`~|{}]\', \' \', text)\n    text = re.sub(r\'[—–]\', \', \', text)\n    text = re.sub(r\'\\s+-\\s+\', \', \', text)\n    text = re.sub(r\'(?m)^\\s*[-•]\\s+\', \'\', text)\n    text = re.sub(r\'\\s+\', \' \', text).strip()\n    return text\n\n\nclass TTSCache:\n    def __init__(self, max_size: int = 100):\n        self._cache: dict[str, bytes] = {}\n        self._max_size = max_size\n\n    def _key(self, text: str) -> str:\n        return hashlib.md5(text.encode()).hexdigest()\n\n    def get(self, text: str) -> bytes | None:\n        return self._cache.get(self._key(text))\n\n    def set(self, text: str, audio: bytes):\n        if len(self._cache) >= self._max_size:\n            oldest = next(iter(self._cache))\n            del self._cache[oldest]\n        self._cache[self._key(text)] = audio\n\n    def clear(self):\n        self._cache.clear()\n\n\nclass AudioService:\n    def __init__(self, model_size: str = "base"):\n        use_gpu = settings.whisper_use_gpu\n        device = "cuda" if use_gpu else "cpu"\n        compute = "float16" if use_gpu else "int8"\n        try:\n            import torch\n            if use_gpu and not torch.cuda.is_available():\n                device = "cpu"\n                compute = "int8"\n        except ImportError:\n            device = "cpu"\n            compute = "int8"\n        self.whisper = WhisperModel(model_size, device=device, compute_type=compute)\n        self.tts_cache = TTSCache(max_size=100)\n\n    async def transcribe(self, audio_data: bytes, filename: str = "audio.webm") -> dict:\n        suffix = Path(filename).suffix or ".webm"\n\n        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:\n            tmp.write(audio_data)\n            tmp_path = tmp.name\n\n        try:\n            def _do_transcribe():\n                segments, info = self.whisper.transcribe(tmp_path, beam_size=1)\n                text = " ".join(seg.text for seg in segments)\n                return text.strip(), info.language, info.duration\n\n            text, lang, duration = await asyncio.to_thread(_do_transcribe)\n            return {"text": text, "language": lang, "duration": duration}\n        finally:\n            Path(tmp_path).unlink(missing_ok=True)\n\n    async def synthesize(self, text: str) -> bytes:\n        text = clean_text_for_tts(text[:500])\n        if not text:\n            return b""\n\n        cached = self.tts_cache.get(text)\n        if cached:\n            return cached\n\n        voices_to_try = [settings.tts_voice, "en-US-AvaNeural", "en-US-ChristopherNeural"]\n        for v in voices_to_try:\n            try:\n                async def _stream_tts(voice_name: str):\n                    communicate = edge_tts.Communicate(text, voice=voice_name, rate=settings.tts_rate, pitch=settings.tts_pitch)\n                    buf = io.BytesIO()\n                    async for chunk in communicate.stream():\n                        if chunk["type"] == "audio":\n                            buf.write(chunk["data"])\n                    buf.seek(0)\n                    return buf.getvalue()\n\n                audio = await asyncio.wait_for(_stream_tts(v), timeout=12.0)\n                if audio and len(audio) > 100:\n                    self.tts_cache.set(text, audio)\n                    return audio\n            except Exception as e:\n                from loguru import logger\n                logger.warning(f"[TTS] Edge-TTS voice {v} failed or timed out: {e}")\n                continue\n\n        return b""\n\n\n    async def synthesize_stream(self, text: str):\n        text = clean_text_for_tts(text[:500])\n        if not text:\n            return\n\n        cached = self.tts_cache.get(text)\n        if cached:\n            yield cached\n            return\n\n        try:\n            communicate = edge_tts.Communicate(text, voice=settings.tts_voice, rate=settings.tts_rate, pitch=settings.tts_pitch)\n            buf = io.BytesIO()\n            async for chunk in communicate.stream():\n                if chunk["type"] == "audio":\n                    buf.write(chunk["data"])\n                    yield chunk["data"]\n            buf.seek(0)\n            audio = buf.getvalue()\n            if audio:\n                self.tts_cache.set(text, audio)\n        except Exception as e:\n            from loguru import logger\n            logger.warning(f"[TTS] Edge-TTS stream failed: {e}")\n'
    with open(audio_path, 'w', encoding='utf-8') as f:
        f.write(patched_audio)
    print('✅ Verified and patched audio_service.py in Colab')

# 4. Configure DEEPSEEK_API_KEY
deepseek_key = ''
try:
    deepseek_key = userdata.get('DEEPSEEK_API_KEY')
except Exception:
    pass

if not deepseek_key:
    deepseek_key = os.environ.get('DEEPSEEK_API_KEY', '').strip()

if not deepseek_key:
    deepseek_key = input('Enter your DEEPSEEK_API_KEY (or press Enter if using Gemini): ').strip()

# 5. Configure GEMINI_API_KEY
gemini_key = ''
try:
    gemini_key = userdata.get('GEMINI_API_KEY')
except Exception:
    pass

if not gemini_key:
    gemini_key = os.environ.get('GEMINI_API_KEY', '').strip()

if not gemini_key and not deepseek_key:
    gemini_key = input('Enter your GEMINI_API_KEY: ').strip()

if deepseek_key:
    os.environ['DEEPSEEK_API_KEY'] = deepseek_key
if gemini_key:
    os.environ['GEMINI_API_KEY'] = gemini_key

provider = 'deepseek' if deepseek_key else 'gemini'
os.environ['LLM_PROVIDER'] = provider

# Write backend/.env
env_path = '/content/Voice_ChatBot/backend/.env'
os.makedirs(os.path.dirname(env_path), exist_ok=True)
with open(env_path, 'w', encoding='utf-8') as f:
    f.write(f'LLM_PROVIDER={provider}\n')
    if deepseek_key:
        f.write(f'DEEPSEEK_API_KEY={deepseek_key}\n')
        f.write('DEEPSEEK_BASE_URL=https://api.deepseek.com\n')
        f.write('DEEPSEEK_MODEL_NAME=deepseek-chat\n')
    if gemini_key:
        f.write(f'GEMINI_API_KEY={gemini_key}\n')
    f.write('VECTOR_STORE_TYPE=chroma\n')

# 6. Terminate old server/tunnel processes
!pkill -f uvicorn
!pkill -f ngrok
ngrok.kill()

# 7. Setup ngrok tunnel
ngrok_token = input('Enter your NGROK_AUTHTOKEN (or press Enter if configured): ').strip()
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

try:
    public_url = ngrok.connect(8000).public_url
    print('\n' + '=' * 70)
    print('🚀 CLOUD GPU BACKEND IS LIVE!')
    print(f'👉 NGROK PUBLIC URL: {public_url}')
    print(f'🤖 ACTIVE LLM PROVIDER: {provider.upper()}')
    print('📋 Copy this URL and set VITE_BACKEND_URL in frontend/.env')
    print('=' * 70 + '\n')
except Exception as e:
    print(f'⚠️ ngrok status: {e}')

# 8. Start Uvicorn backend server
%cd /content/Voice_ChatBot/backend
!python -m uvicorn app.main:app --host 0.0.0.0 --port 8000

✅ Verified and patched config.py in Colab
✅ Verified and patched llm.py in Colab
✅ Verified and patched audio_service.py in Colab
Enter your DEEPSEEK_API_KEY (or press Enter if using Gemini): ***REMOVED***
Enter your NGROK_AUTHTOKEN (or press Enter if configured): ***REMOVED***

🚀 CLOUD GPU BACKEND IS LIVE!
👉 NGROK PUBLIC URL: https://hash-phoniness-freely.ngrok-free.dev
🤖 ACTIVE LLM PROVIDER: DEEPSEEK
📋 Copy this URL and set VITE_BACKEND_URL in frontend/.env

/content/Voice_ChatBot/backend
INFO:     Started server process [2065]
INFO:     Waiting for application startup.
2026-08-03 16:56:36.999 | INFO     | app.main:lifespan:34 - Starting up: loading models and stores...
2026-08-03 16:56:36.999 | INFO     | app.main:lifespan:41 - Loading embedding model: BAAI/bge-m3
modules.json: 100% 349/349 [00:00<00:00, 1.76MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 877kB/s]
README.md: 100% 15.8k/15.8k [00:00<00:00, 47.5MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00